## Get DFT features

In [ ]:
from __future__ import annotations

from pathlib import Path

import cclib
import numpy as np
import pandas as pd
from rdkit import Chem

ATOM = {
    1: "H",
    5: "B",
    6: "C",
    7: "N",
    8: "O",
    9: "F",
    14: "Si",
    15: "P",
    16: "S",
    17: "Cl",
    35: "Br",
}

# ────Indices of structure──────────────────────────────────────────────────────────────────────────────


def acid_idx(row: pd.Series) -> tuple:
    smiles = row["smiles"]
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    # Detect the indices of the atoms in the phthalic acid scaffold.
    carboxyl_pattern= Chem.MolFromSmarts("[CX3](=[OX1])[OX2H1][H]")
    sulfonic_pattern = Chem.MolFromSmarts("[S](=[OX1])(=[OX1])[OX2H1][H]")
    c_matches = mol.GetSubstructMatches(carboxyl_pattern)
    s_matches = mol.GetSubstructMatches(sulfonic_pattern)
    if len(s_matches) > 0:
        return s_matches
    else:
        return c_matches

# ────Descritors──────────────────────────────────────────────────────────────────────────────


def homolumo(logdata: cclib.io.ccread) -> dict:
    homolumo_idx = logdata.homos[0]
    homo_ev = logdata.moenergies[0][homolumo_idx]
    lumo_ev = logdata.moenergies[0][homolumo_idx + 1]
    gap_ev = lumo_ev - homo_ev
    return {
        "homo_ev": homo_ev,
        "lumo_ev": lumo_ev,
        "gap_ev": gap_ev,
    }


def dipole_moment(logdata: cclib.io.ccread) -> dict:
    vector = logdata.moments[1]
    scalar = np.linalg.norm(vector)
    return {"dipole_moment_debye": scalar}


def get_final_nbo_charges(logfile_path: Path, acid_sites: list[list[int]]) -> dict:
    # NBO charge is obtained the value before the structure optimization when using cclib, so search from the end.
    acid_h = []
    acid_o = []
    for acid_site in len(acid_sites):
        acid_h.append(acid_site[-1])
        acid_o.append(acid_site[-2])

    with open(logfile_path, "r") as f:
        lines = f.readlines()

    start_line = -1
    for i in range(len(lines) - 1, -1, -1):
        if "Summary of Natural Population Analysis" in lines[i]:
            start_line = i
            break

    if start_line == -1:
        return None

    charges = []
    for line in lines[start_line + 6 :]:
        # Skip until the NBO charges information appears.
        if "---" in line or "Total" in line:
            break
        parts = line.split()
        if len(parts) >= 3:
            charges.append(float(parts[2]))
    h_nbo = []
    o_nbo = []
    total = []
    for i in range(len(acid_sites)):
        h_nbo.append(charges[acid_h[i]])
        o_nbo.append(charges[acid_o[i]])
        total_num = 0
        for j in acid_sites[i]:
            total_num += charges[j]
        total.append(total_num)

    max_h_nbo = max(h_nbo)
    min_o_nbo = min(o_nbo)
    avg_total = sum(total) / len(total)

    return {"h_nbo_charge": max_h_nbo, "o_nbo_charge": min_o_nbo, "total_nbo_charge": avg_total}


def polar(logdata: cclib.io.ccread) -> dict:
    tensor = logdata.polarizabilities[-1]

    # mean polarizability (alpha_iso) = (XX + YY + ZZ) / 3
    # Using NumPy's "trace" function makes it easy to calculate the sum of diagonal elements (XX + YY + ZZ).
    alpha_iso = np.trace(tensor) / 3
    return {"polar": alpha_iso}


def calc_features(
    logfile_path: Path,
    logdata: cclib.io.ccread,
    acid_sites: list[list[int]],
) -> dict:
    result = {}
    result.update(polar(logdata))
    result.update(get_final_nbo_charges(logfile_path, acid_sites))
    result.update(dipole_moment(logdata))
    result.update(homolumo(logdata))
    return result


# ────Main function──────────────────────────────────────────────────────────────────────────────


def get_data(
    df: pd.DataFrame, name: str, known: bool = True
) -> pd.DataFrame:
    base_path = Path(__file__).resolve().parent.parent.parent / "data/in/"
    result = []
    valid_indices = []
    invalid_indices = []
    for idx, row in df.iterrows():  # df can not use for loop as is, so use iterrows().
        cas = row["cas"]
        acid_sites = acid_idx(row)
        # The directory is different for known and unknown data.
        path_pattern = (
            f"known/sub_{cas}.log" if known else f"unknown/*/logfile/sub_{cas}.log"
        )
        logfile_path = list(base_path.glob(path_pattern))
        if len(logfile_path) == 0:
            invalid_indices.append(idx)
            continue
        # Check if the file exists.
        if not logfile_path[0].is_file():
            invalid_indices.append(idx)
            continue
        logfile_path = logfile_path[0]
        logdata = cclib.io.ccread(str(logfile_path))
        result.append(
            calc_features(
                logfile_path,
                logdata,
                acid_sites,
            )
        )
        valid_indices.append(idx)

    df_valid = df.loc[valid_indices].reset_index(drop=True)
    df_invalid = df.loc[invalid_indices].reset_index(drop=True)
    features_df = pd.DataFrame(result).reset_index(drop=True)
    df_valid = pd.concat([df_valid.reset_index(drop=True), features_df], axis=1)

    if known:
        df_valid.to_csv(f"../data/known/features/{name}.csv", index=False)
        df_invalid.to_csv(f"../data/known/features/{name}_invalid.csv", index=False)
    elif not known:
        df_valid.to_csv(f"../data/unknown/features/{name}.csv", index=False)
        df_invalid.to_csv(f"../data/unknown/features/{name}_invalid.csv", index=False)

    return df_valid


In [10]:
from rdkit import Chem

smiles = "OC(=O)C1=CC=CC=C1C(=O)O"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# Detect the indices of the atoms in the phthalic acid scaffold.
carboxyl_pattern= Chem.MolFromSmarts("[CX3](=[OX1])[OX2H1][H]")
sulfonic_pattern = Chem.MolFromSmarts("[S](=[OX1])(=[OX1])[OX2H1][H]")
matches_tuple = mol.GetSubstructMatches(carboxyl_pattern)
matches_tuple_v2 = mol.GetSubstructMatches(sulfonic_pattern)
print(len(matches_tuple))
print(len(matches_tuple_v2))
for match in matches_tuple:
    print(match[-1])

2
0
12
17
